# Control Engineer Onboarding (15 minutes)

A hands-on tour of the SCPN Phase Orchestrator (SPO). By the end you will have:

1. **Validated** a domain *binding spec* — the contract that maps your signals
   onto a network of coupled oscillators.
2. **Run** the phase-dynamics engine and read the global coherence `R` — the
   universal synchronisation signal (`0` = incoherent, `1` = fully locked).
3. Seen that **coupling strength is the control knob**: below a critical value the
   network desynchronises and `R` collapses; a supervisor that boosts coupling
   past the threshold restores coherence.
4. **Audited and replayed** a run — every step is SHA-256 hash-chained for
   deterministic, tamper-evident reproducibility.

Everything uses the public Python API and the pure-Python path, so it runs from a
bare `pip install scpn-phase-orchestrator`. The same golden path is one CLI line:
`spo quickstart power`.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from scpn_phase_orchestrator.binding import load_binding_spec, validate_binding_spec
from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.runtime.audit_logger import AuditLogger
from scpn_phase_orchestrator.runtime.replay import ReplayEngine
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.metrics import LayerState, UPDEState
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

## 1. Validate a binding spec

A *binding spec* is the domain contract: it declares the oscillator layers,
coupling template, safety tier, and actuators for a domain. Validation is
fail-closed and collects every problem at once, so you never run an ill-formed
domain. Here we load and validate a shipped example.

In [ ]:
# nbconvert runs notebooks from the notebooks/ directory, so repo paths
# are relative to it (matching the other domainpack notebooks).
spec_path = Path("../domainpacks/minimal_domain/binding_spec.yaml")
spec = load_binding_spec(spec_path)
errors = validate_binding_spec(spec)
print(f"binding: {spec.name} v{spec.version} (safety_tier={spec.safety_tier})")
if errors:
    print(f"validation: {len(errors)} error(s): {errors}")
else:
    print("validation: valid - no errors")

## 2. `R` is the health signal; coupling is the control knob

The engine integrates a network of heterogeneous oscillators. The global order
parameter `R` measures how phase-aligned they are. With too little coupling the
oscillators drift apart and `R` stays near zero; past a critical coupling they
lock and `R` approaches one. We sweep the coupling strength to see the
synchronisation transition — the response surface a supervisor exploits.

In [ ]:
N_OSC, DT, STEPS = 12, 0.01, 1000
rng = np.random.default_rng(7)
omegas = rng.normal(1.0, 0.45, N_OSC)
template = CouplingBuilder().build(N_OSC, base_strength=1.0, decay_alpha=0.25)
alpha = template.alpha.copy()
phases_init = rng.uniform(0, 2 * np.pi, N_OSC)


def run_to_steady_R(coupling_strength):
    engine = UPDEEngine(N_OSC, DT)
    phases = phases_init.copy()
    knm = template.knm * coupling_strength
    R = 0.0
    for _ in range(STEPS):
        phases = engine.step(phases, omegas, knm, 0.0, 0.0, alpha)
        R, _ = compute_order_parameter(phases)
    return R


strengths = [0.05, 0.15, 0.35, 0.70, 1.40]
final_R = [run_to_steady_R(k) for k in strengths]
for k, r in zip(strengths, final_R, strict=True):
    print(f"coupling K={k:.2f}  ->  steady R={r:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(strengths, final_R, "o-", color="#1f77b4", lw=1.5)
ax.axhline(0.9, color="#999999", ls=":", lw=1, label="coherent threshold")
ax.set_xlabel("coupling strength K")
ax.set_ylabel("steady-state R")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Synchronisation transition: coherence responds to coupling")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

### Reading the transition

Below `K ~ 0.3` the network is desynchronised (`R` near zero) — the "disturbed"
regime. Above it the oscillators lock (`R ~ 1`). A supervisor detecting a low `R`
responds by boosting coupling above the threshold; that single knob is the
recovery lever behind the domainpacks' policy rules.

## 3. Audit and replay — deterministic, tamper-evident runs

Every step of a run is written to a SHA-256 hash-chained JSONL audit trail. The
trail lets you (a) verify no entry was altered and (b) replay the run
bit-for-bit from its logged initial conditions. We run one healthy domain
(`K = 0.70`), audit it, then verify integrity and determinism.

In [ ]:
K_HEALTHY = 0.70
audit_path = Path(tempfile.mkdtemp()) / "onboarding_audit.jsonl"
logger = AuditLogger(path=audit_path)
logger.log_header(n_oscillators=N_OSC, dt=DT, method="euler", seed=7)

engine = UPDEEngine(N_OSC, DT)
phases = phases_init.copy()
knm = template.knm * K_HEALTHY
R_trace = []
for step in range(500):
    phases = engine.step(phases, omegas, knm, 0.0, 0.0, alpha)
    R, psi = compute_order_parameter(phases)
    R_trace.append(R)
    state = UPDEState(
        layers=[LayerState(R=R, psi=psi)],
        cross_layer_alignment=np.zeros((1, 1)),
        stability_proxy=R,
        regime_id="nominal",
    )
    logger.log_step(
        step, state, actions=[], phases=phases, omegas=omegas, knm=knm, alpha=alpha
    )
logger.close()

replay = ReplayEngine(audit_path)
entries = replay.load()
header = replay.load_header(entries)
integrity_ok, n_checked = ReplayEngine.verify_integrity(entries)
determinism_ok, n_replayed = replay.verify_determinism_chained(
    replay.build_engine(header), entries, atol=1e-10
)
print(f"logged steps:         {len(entries) - 1}")
print(f"hash-chain intact:    {integrity_ok} ({n_checked} entries)")
print(f"replay deterministic: {determinism_ok} ({n_replayed} steps)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(np.arange(len(R_trace)) * DT, R_trace, color="#2ca02c", lw=1.2)
ax.set_xlabel("time (s)")
ax.set_ylabel("R")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Healthy run (K=0.70): coherence converges and holds")
fig.tight_layout()
plt.show()

## What you learned & where to go next

- A **binding spec** is the validated domain contract (`load_binding_spec` +
  `validate_binding_spec`).
- **`R`** is the universal coherence signal; **coupling strength** is the knob
  that moves the system across the synchronisation transition.
- Runs are **auditable and replayable** — SHA-256 hash-chained and deterministic.

Next steps:

- **Choosing an engine** — the engine shortlist and honest evidence caveat:
  [choosing_an_engine.md](../docs/guide/choosing_an_engine.md).
- **Install profiles & the PyPI boundary**:
  [install_profiles.md](../docs/guide/install_profiles.md).
- **From notebook to production**:
  [notebook_to_production.md](../docs/guide/notebook_to_production.md).
- Run the whole golden path from the CLI: `spo quickstart power --steps 200`.